# Neural Receiver - Complete Simulation Pipeline

This notebook runs the complete simulation pipeline for weak signal detection and parametric classification.

## Pipeline Overview
1. **Setup**: Install dependencies and import modules
2. **Signal Generation**: Generate synthetic IQ signals with various modulation types
3. **Model Training**: Train the multi-task neural receiver
4. **Evaluation**: Evaluate performance on test data
5. **Visualization**: Display results and metrics

## 1. Setup and Installation

In [ ]:
# Check if running in Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    # Install required packages
    !pip install -q torch torchvision tensorboard numpy scipy scikit-learn matplotlib seaborn tqdm pyyaml
else:
    print("Running in local environment")
    print("Make sure you have installed requirements: pip install -r requirements.txt")

In [ ]:
# Import all necessary modules
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path
import json
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [ ]:
# Import project modules
from src.data import SignalGenerator, SignalDataset, SignalParams
from src.models import NeuralReceiver
from src.training import Trainer
from src.evaluation import Evaluator, MetricsCalculator
from src.utils import plot_signal_examples, plot_training_history, plot_confusion_matrix

print("All modules imported successfully!")

## 2. Signal Generation Demo

First, let's generate and visualize some example signals.

In [ ]:
# Initialize signal generator
signal_gen = SignalGenerator(sample_rate=1.0, seed=42)

# Modulation types to demonstrate
modulation_types = ['No-signal', 'AM', 'FM', '2FSK', 'CW Radar', 'LFM/Chirp']
sequence_length = 1024
snr_db = -5.0  # Weak signal scenario

print(f"Generating {len(modulation_types)} signal examples at SNR = {snr_db} dB")
print("="*70)

# Generate example signals
signals = []
for mod_type in modulation_types:
    params = signal_gen.generate_random_params(
        modulation_type=mod_type,
        snr_range=(snr_db, snr_db)
    )
    signal, noisy_signal = signal_gen.generate_signal(sequence_length, params)
    signals.append((mod_type, noisy_signal, params))
    print(f"✓ Generated {mod_type:20s} - f_c={params.center_freq:.3f}, BW={params.bandwidth:.3f}")

print("="*70)

In [ ]:
# Visualize signal examples
fig, axes = plt.subplots(len(modulation_types), 2, figsize=(15, 3*len(modulation_types)))

for idx, (mod_type, signal, params) in enumerate(signals):
    # Time domain
    ax1 = axes[idx, 0]
    time = np.arange(len(signal))
    ax1.plot(time[:200], signal.real[:200], label='I', alpha=0.7)
    ax1.plot(time[:200], signal.imag[:200], label='Q', alpha=0.7)
    ax1.set_title(f'{mod_type} - Time Domain (SNR={snr_db} dB)')
    ax1.set_xlabel('Sample')
    ax1.set_ylabel('Amplitude')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Frequency domain
    ax2 = axes[idx, 1]
    fft = np.fft.fftshift(np.fft.fft(signal))
    freq = np.fft.fftshift(np.fft.fftfreq(len(signal)))
    ax2.plot(freq, 20*np.log10(np.abs(fft) + 1e-10))
    ax2.set_title(f'{mod_type} - Frequency Domain')
    ax2.set_xlabel('Normalized Frequency')
    ax2.set_ylabel('Magnitude (dB)')
    ax2.grid(True, alpha=0.3)
    ax2.axvline(params.center_freq, color='r', linestyle='--', alpha=0.5, label='Center Freq')
    ax2.legend()

plt.tight_layout()
plt.savefig('signal_examples.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSignal examples saved to 'signal_examples.png'")

## 3. Dataset Preparation

Create training, validation, and test datasets.

In [ ]:
# Configuration for dataset generation
config = {
    'sequence_length': 1024,
    'snr_range': (-10, 0),  # Weak signal range
    'train_samples': 10000,
    'val_samples': 2000,
    'test_samples': 2000,
    'batch_size': 64,
    'num_workers': 2
}

print("Dataset Configuration:")
print("="*70)
for key, value in config.items():
    print(f"  {key:20s}: {value}")
print("="*70)

In [ ]:
# Create datasets
print("\nGenerating datasets...")

train_dataset = SignalDataset(
    num_samples=config['train_samples'],
    sequence_length=config['sequence_length'],
    snr_range=config['snr_range'],
    seed=42
)

val_dataset = SignalDataset(
    num_samples=config['val_samples'],
    sequence_length=config['sequence_length'],
    snr_range=config['snr_range'],
    seed=43
)

test_dataset = SignalDataset(
    num_samples=config['test_samples'],
    sequence_length=config['sequence_length'],
    snr_range=config['snr_range'],
    seed=44
)

print(f"✓ Training set:   {len(train_dataset):6d} samples")
print(f"✓ Validation set: {len(val_dataset):6d} samples")
print(f"✓ Test set:       {len(test_dataset):6d} samples")

# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=config['num_workers']
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['num_workers']
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['num_workers']
)

print(f"✓ Data loaders created (batch_size={config['batch_size']})")

In [ ]:
# Check dataset statistics
print("\nDataset Statistics:")
print("="*70)

# Get modulation type distribution
mod_types = [train_dataset.modulation_classes[i] for i in range(len(train_dataset.modulation_classes))]
print(f"Number of modulation classes: {len(mod_types)}")
print(f"Modulation types: {', '.join(mod_types)}")

# Check a sample
sample = train_dataset[0]
print(f"\nSample structure:")
print(f"  IQ signal shape: {sample['iq'].shape}")
print(f"  Has signal: {sample['has_signal']}")
print(f"  Modulation class: {sample['modulation_class']}")
print(f"  Parameters shape: {sample['parameters'].shape}")
print("="*70)

## 4. Model Creation

Create the multi-task neural receiver model.

In [ ]:
# Model configuration
model_config = {
    'input_channels': 2,  # I and Q
    'base_channels': 64,
    'num_blocks': 4,
    'num_classes': len(train_dataset.modulation_classes),
    'num_params': 4,  # center_freq, bandwidth, power, symbol_rate
    'dropout': 0.3
}

print("Model Configuration:")
print("="*70)
for key, value in model_config.items():
    print(f"  {key:20s}: {value}")
print("="*70)

In [ ]:
# Create model
model = NeuralReceiver(**model_config).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel created successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

## 5. Training

Train the model using the multi-task learning framework.

In [ ]:
# Training configuration
training_config = {
    'num_epochs': 50,
    'learning_rate': 0.001,
    'weight_decay': 1e-5,
    'save_dir': 'experiments/neural_receiver',
    'loss_weights': {
        'detection': 1.0,
        'classification': 1.0,
        'regression': 1.0
    }
}

print("Training Configuration:")
print("="*70)
for key, value in training_config.items():
    print(f"  {key:20s}: {value}")
print("="*70)

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    learning_rate=training_config['learning_rate'],
    weight_decay=training_config['weight_decay'],
    save_dir=training_config['save_dir'],
    loss_weights=training_config['loss_weights']
)

print("Trainer initialized successfully!")

In [ ]:
# Train the model
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70 + "\n")

history = trainer.train(num_epochs=training_config['num_epochs'])

print("\n" + "="*70)
print("TRAINING COMPLETED")
print("="*70)

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Total loss
axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Total Loss')
axes[0, 0].set_title('Total Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Detection accuracy
axes[0, 1].plot(history['train_detection_acc'], label='Train', linewidth=2)
axes[0, 1].plot(history['val_detection_acc'], label='Validation', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Detection Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Classification accuracy
axes[1, 0].plot(history['train_classification_acc'], label='Train', linewidth=2)
axes[1, 0].plot(history['val_classification_acc'], label='Validation', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('Modulation Classification Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Regression MAE
axes[1, 1].plot(history['train_regression_mae'], label='Train', linewidth=2)
axes[1, 1].plot(history['val_regression_mae'], label='Validation', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('MAE')
axes[1, 1].set_title('Parameter Estimation MAE')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("Training history saved to 'training_history.png'")

## 6. Evaluation

Evaluate the trained model on the test set.

In [ ]:
# Load best model
best_model_path = Path(training_config['save_dir']) / 'checkpoints' / 'best_model.pt'
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']}")
print(f"Best validation loss: {checkpoint['val_loss']:.4f}")

In [ ]:
# Create evaluator
evaluator = Evaluator(
    model=model,
    test_loader=test_loader,
    device=device,
    class_names=train_dataset.modulation_classes
)

print("\n" + "="*70)
print("STARTING EVALUATION")
print("="*70 + "\n")

# Run evaluation
results = evaluator.evaluate()

print("\n" + "="*70)
print("EVALUATION COMPLETED")
print("="*70)

In [ ]:
# Print detailed metrics
print("\n" + "="*70)
print("DETECTION METRICS")
print("="*70)
print(f"ROC AUC:        {results['detection']['roc_auc']:.4f}")
print(f"Accuracy:       {results['detection']['accuracy']:.4f}")
print(f"Precision:      {results['detection']['precision']:.4f}")
print(f"Recall:         {results['detection']['recall']:.4f}")
print(f"F1 Score:       {results['detection']['f1']:.4f}")

print("\n" + "="*70)
print("CLASSIFICATION METRICS")
print("="*70)
print(f"Accuracy:       {results['classification']['accuracy']:.4f}")
print(f"Macro F1:       {results['classification']['f1_macro']:.4f}")
print(f"Weighted F1:    {results['classification']['f1_weighted']:.4f}")

print("\n" + "="*70)
print("REGRESSION METRICS")
print("="*70)
print(f"Overall MAE:    {results['regression']['mae']:.4f}")
print(f"Overall RMSE:   {results['regression']['rmse']:.4f}")
print(f"\nPer-parameter MAE:")
param_names = ['Center Freq', 'Bandwidth', 'SNR', 'Symbol Rate']
for name, mae in zip(param_names, results['regression']['mae_per_param']):
    print(f"  {name:15s}: {mae:.4f}")

## 7. Visualization of Results

In [ ]:
# Plot ROC curve
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ROC Curve
axes[0].plot(results['detection']['fpr'], results['detection']['tpr'], 
             linewidth=2, label=f"ROC (AUC = {results['detection']['roc_auc']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve - Signal Detection')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
axes[1].plot(results['detection']['recall_curve'], results['detection']['precision_curve'],
             linewidth=2, label=f"PR Curve")
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve - Signal Detection')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('detection_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("Detection metrics saved to 'detection_metrics.png'")

In [ ]:
# Plot confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))

cm = results['classification']['confusion_matrix']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=train_dataset.modulation_classes,
            yticklabels=train_dataset.modulation_classes,
            ax=ax, cbar_kws={'label': 'Count'})

ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix - Modulation Classification')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Confusion matrix saved to 'confusion_matrix.png'")

In [ ]:
# Plot regression errors
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
param_names = ['Center Frequency', 'Bandwidth', 'SNR (dB)', 'Symbol Rate']

predictions = results['regression']['predictions']
targets = results['regression']['targets']

for idx, (ax, param_name) in enumerate(zip(axes.flatten(), param_names)):
    pred = predictions[:, idx]
    true = targets[:, idx]
    
    # Scatter plot
    ax.scatter(true, pred, alpha=0.3, s=10)
    
    # Perfect prediction line
    min_val = min(true.min(), pred.min())
    max_val = max(true.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
    
    # Calculate error metrics
    mae = np.mean(np.abs(pred - true))
    rmse = np.sqrt(np.mean((pred - true)**2))
    
    ax.set_xlabel(f'True {param_name}')
    ax.set_ylabel(f'Predicted {param_name}')
    ax.set_title(f'{param_name}\nMAE: {mae:.4f}, RMSE: {rmse:.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('regression_errors.png', dpi=150, bbox_inches='tight')
plt.show()

print("Regression errors saved to 'regression_errors.png'")

## 8. Inference Demo

Test the model on individual samples.

In [ ]:
# Generate a test signal
test_mod = 'FM'
test_snr = -8.0

test_params = signal_gen.generate_random_params(
    modulation_type=test_mod,
    snr_range=(test_snr, test_snr)
)
test_signal, test_noisy = signal_gen.generate_signal(sequence_length, test_params)

print(f"Generated test signal:")
print(f"  Modulation: {test_mod}")
print(f"  SNR: {test_snr} dB")
print(f"  Center Freq: {test_params.center_freq:.4f}")
print(f"  Bandwidth: {test_params.bandwidth:.4f}")
print(f"  Symbol Rate: {test_params.symbol_rate:.1f}")

In [ ]:
# Run inference
iq_tensor = torch.stack([
    torch.from_numpy(test_noisy.real).float(),
    torch.from_numpy(test_noisy.imag).float()
]).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(iq_tensor)
    predictions = model.predict(iq_tensor)

print("\n" + "="*70)
print("INFERENCE RESULTS")
print("="*70)
print(f"Signal Detected:     {predictions['signal_detected'].item()}")
print(f"Detection Prob:      {predictions['detection_prob'].item():.4f}")
print(f"Modulation Class:    {train_dataset.modulation_classes[predictions['modulation_class'].item()]}")
print(f"Classification Conf: {predictions['classification_confidence'].item():.4f}")
print(f"\nEstimated Parameters:")
print(f"  Center Frequency:  {predictions['parameters'][0, 0].item():.4f} (true: {test_params.center_freq:.4f})")
print(f"  Bandwidth:         {predictions['parameters'][0, 1].item():.4f} (true: {test_params.bandwidth:.4f})")
print(f"  SNR:               {predictions['parameters'][0, 2].item():.2f} dB (true: {test_snr:.2f} dB)")
print(f"  Symbol Rate:       {predictions['parameters'][0, 3].item():.1f} (true: {test_params.symbol_rate:.1f})")
print("="*70)

## 9. Summary

Complete simulation pipeline executed successfully!

In [ ]:
# Save final results
final_results = {
    'config': {
        'dataset': config,
        'model': model_config,
        'training': training_config
    },
    'metrics': {
        'detection': {
            'roc_auc': float(results['detection']['roc_auc']),
            'accuracy': float(results['detection']['accuracy']),
            'f1': float(results['detection']['f1'])
        },
        'classification': {
            'accuracy': float(results['classification']['accuracy']),
            'f1_macro': float(results['classification']['f1_macro']),
            'f1_weighted': float(results['classification']['f1_weighted'])
        },
        'regression': {
            'mae': float(results['regression']['mae']),
            'rmse': float(results['regression']['rmse'])
        }
    }
}

with open('simulation_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("\n" + "="*70)
print("SIMULATION COMPLETE")
print("="*70)
print("\nResults saved to:")
print("  - simulation_results.json")
print("  - signal_examples.png")
print("  - training_history.png")
print("  - detection_metrics.png")
print("  - confusion_matrix.png")
print("  - regression_errors.png")
print(f"\nModel checkpoint: {best_model_path}")
print("="*70)

In [ ]:
# Display final summary
print("\n" + "#" * 70)
print("#" + " " * 68 + "#")
print("#" + " " * 15 + "SIMULATION SUMMARY" + " " * 35 + "#")
print("#" + " " * 68 + "#")
print("#" * 70)
print(f"\n{'Dataset':<30s} {config['train_samples']:>10,} training samples")
print(f"{'SNR Range':<30s} {config['snr_range'][0]:>6.1f} to {config['snr_range'][1]:>6.1f} dB")
print(f"{'Sequence Length':<30s} {config['sequence_length']:>10,} samples")
print(f"\n{'Model Architecture':<30s}")
print(f"  {'Total Parameters':<28s} {total_params:>10,}")
print(f"  {'Base Channels':<28s} {model_config['base_channels']:>10}")
print(f"  {'Number of Blocks':<28s} {model_config['num_blocks']:>10}")
print(f"\n{'Training':<30s}")
print(f"  {'Epochs':<28s} {training_config['num_epochs']:>10}")
print(f"  {'Learning Rate':<28s} {training_config['learning_rate']:>10.6f}")
print(f"  {'Batch Size':<28s} {config['batch_size']:>10}")
print(f"\n{'Performance Metrics':<30s}")
print(f"  {'Detection ROC-AUC':<28s} {results['detection']['roc_auc']:>10.4f}")
print(f"  {'Classification Accuracy':<28s} {results['classification']['accuracy']:>10.4f}")
print(f"  {'Parameter MAE':<28s} {results['regression']['mae']:>10.4f}")
print("\n" + "#" * 70)
print("\nThank you for using Neural Receiver!")
print("For more information, see README.md and QUICKSTART.md")
print("#" * 70)